# 05 — GISMo + Filter Baseline (post-hoc, no training)

Loads the vanilla GISMo checkpoint trained in `02_train_baseline.ipynb` and applies post-hoc health filters (hard / soft with alpha sweep). This is the strongest non-trained baseline -- our trained methods need to beat both vanilla GISMo and this filtered variant.

**Prerequisite**: `02_train_baseline.ipynb` must have been run (produces `best_baseline.pt`).

**Runtime**: Colab free T4 GPU, ~10 min.

**Steps**:
1. Runtime > Change runtime type > T4 GPU
2. Set `PROJECT_ROOT` (cell 5) to wherever you put `data/` in your Drive
3. Runtime > Run all

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
# torch_geometric needs to be matched with the installed torch wheel.
# Colab's default torch is recent enough that the generic install works.
!pip install -q torch_geometric pandas numpy matplotlib

## 2. Get the code

In [ ]:
# Clone (or refresh) the repo
import os
if not os.path.exists('/content/CS471-Project'):
    !git clone https://github.com/lee1june61/CS471-Project.git /content/CS471-Project
else:
    !cd /content/CS471-Project && git pull --ff-only
%cd /content/CS471-Project

## 3. Mount Drive for data + outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === EDIT THIS if your Drive layout is different ===
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
DATA_DIR     = f'{PROJECT_ROOT}/data'
OUTPUT_DIR   = f'{PROJECT_ROOT}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# sanity check
expected = ['flavorgraph_edges.csv', 'nodes_filtered.csv',
             'pairs_train.csv', 'pairs_val.csv', 'pairs_test.csv',
             'recipes.json', 'usda_mapping.json']
missing = [f for f in expected if not os.path.exists(f'{DATA_DIR}/{f}')]
if missing:
    raise FileNotFoundError(f'missing data files in {DATA_DIR}: {missing}')
print(f'DATA_DIR   = {DATA_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

## 4. Run post-hoc filters

In [ ]:
ckpt_path = f'{OUTPUT_DIR}/baseline/best_baseline.pt'
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f'{ckpt_path} not found. Run 02_train_baseline.ipynb first.')

!python src/eval_filter_baseline.py \
  --checkpoint {ckpt_path} \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/filter_baseline \
  --filter_mode both --alpha 0.5 1.0

## 5. Summary

In [ ]:
import json
with open(f'{OUTPUT_DIR}/filter_baseline/summary.json') as f:
    summary = json.load(f)
print(f'{"config":<35} {"MRR":>7} {"Hit@1":>7} {"Hit@10":>7}')
print('-' * 65)
for k in sorted(summary.keys()):
    m = summary[k]
    print(f'{k:<35} {m["MRR"]:>7.2f} {m["Hit@1"]:>7.2f} {m["Hit@10"]:>7.2f}')